# Binance Options Partial Depth Service

This notebook shows how to call the local `BinanceOptionsDepthService` module from Jupyter. It loads official Binance Options symbols through `/eapi/v1/exchangeInfo`, selects a small set of currently trading contracts, subscribes to Options partial depth streams, and reads bid1/bid2/ask1/ask2 through `get_latest_level_quote()`.

Run the cleanup cell at the end when you are done because the depth service starts a background receiver thread.

## 1. Import Local Package

In [1]:
import sys
import time
from decimal import Decimal
from pathlib import Path

import pandas as pd
from IPython.display import clear_output, display


def find_repo_path():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path("/home/suncong/binance_klines_data_fetch"),
    ]
    for candidate in candidates:
        if (candidate / "binance_klines_data_fetch").is_dir():
            return candidate.resolve()
    raise RuntimeError("Could not find the binance_klines_data_fetch repo path")


repo_path = find_repo_path()
repo_path_str = str(repo_path)
if repo_path_str not in sys.path:
    sys.path.insert(0, repo_path_str)

from binance_klines_data_fetch import (
    BinanceOptionsDepthConfig,
    BinanceOptionsDepthService,
    build_options_depth_service_configs,
    filter_options,
    get_option_universe,
    select_near_atm_options,
    select_nearest_expiries,
)

print("Imported package from:", repo_path)

Imported package from: /home/suncong/binance_klines_data_fetch


## 2. Choose Options Contracts

Options symbols are dynamic. Do not hand-build them for production use. This cell loads Binance's official Options universe, filters active BTCUSDT contracts, selects the nearest expiry, then keeps strikes near a reference price. Replace `REFERENCE_PRICE` with your current index or spot price when running the notebook.

In [2]:
UNDERLYING = "BTCUSDT"
REFERENCE_PRICE = Decimal("73764.65")
N_STRIKES_EACH_SIDE = 2
MAX_CONTRACTS_FOR_EXAMPLE = 12

universe = get_option_universe()
trading_options = filter_options(
    universe,
    underlying=UNDERLYING,
    status="TRADING",
)

if not trading_options:
    raise RuntimeError(f"No TRADING options found for {UNDERLYING}")

nearest_expiry = select_nearest_expiries(trading_options, count=1)[0]
near_expiry_options = filter_options(trading_options, expiry=nearest_expiry)
selected_options = select_near_atm_options(
    near_expiry_options,
    spot=REFERENCE_PRICE,
    n_strikes_each_side=N_STRIKES_EACH_SIDE,
)[:MAX_CONTRACTS_FOR_EXAMPLE]

if not selected_options:
    raise RuntimeError("No options selected for depth subscription")

selection_df = pd.DataFrame(
    [
        {
            "symbol": opt.symbol,
            "underlying": opt.underlying,
            "expiry": opt.expiry,
            "strike": opt.strike,
            "type": opt.option_type,
            "status": opt.status,
        }
        for opt in selected_options
    ]
)

print(f"Loaded {len(universe)} option symbols")
print(f"{UNDERLYING} TRADING contracts: {len(trading_options)}")
print("Nearest expiry:", nearest_expiry)
display(selection_df)

Loaded 1628 option symbols
BTCUSDT TRADING contracts: 566
Nearest expiry: 2026-06-01 08:00:00+00:00


,symbol,underlying,expiry,strike,type,status
0,BTC-260601-73000-C,BTCUSDT,2026-06-01 08:00:00+00:00,73000.000,CALL,TRADING
1,BTC-260601-73000-P,BTCUSDT,2026-06-01 08:00:00+00:00,73000.000,PUT,TRADING
2,BTC-260601-73500-C,BTCUSDT,2026-06-01 08:00:00+00:00,73500.000,CALL,TRADING
3,BTC-260601-73500-P,BTCUSDT,2026-06-01 08:00:00+00:00,73500.000,PUT,TRADING
4,BTC-260601-74000-C,BTCUSDT,2026-06-01 08:00:00+00:00,74000.000,CALL,TRADING
5,BTC-260601-74000-P,BTCUSDT,2026-06-01 08:00:00+00:00,74000.000,PUT,TRADING
6,BTC-260601-74500-C,BTCUSDT,2026-06-01 08:00:00+00:00,74500.000,CALL,TRADING
7,BTC-260601-74500-P,BTCUSDT,2026-06-01 08:00:00+00:00,74500.000,PUT,TRADING
8,BTC-260601-75000-C,BTCUSDT,2026-06-01 08:00:00+00:00,75000.000,CALL,TRADING
9,BTC-260601-75000-P,BTCUSDT,2026-06-01 08:00:00+00:00,75000.000,PUT,TRADING


## 3. Build Options Depth Configuration

Binance Options partial depth supports levels `5`, `10`, and `20`; update speed supports `100ms` and `500ms`. A single Options WebSocket connection can listen to at most 200 streams. The helper below returns one config for this small example and would split larger symbol lists into multiple configs.

In [3]:
LEVELS = 5
SPEED_MS = 500

configs = build_options_depth_service_configs(
    selected_options,
    levels=LEVELS,
    speed_ms=SPEED_MS,
    read_timeout_seconds=120.0,
    startup_timeout_seconds=60.0,
)

print("Number of websocket configs:", len(configs))
print("Symbols in first config:", tuple(configs[0].symbols))

service = BinanceOptionsDepthService(configs[0])
print(service.build_url())

Number of websocket configs: 1
Symbols in first config: ('BTC-260601-73000-C', 'BTC-260601-73000-P', 'BTC-260601-73500-C', 'BTC-260601-73500-P', 'BTC-260601-74000-C', 'BTC-260601-74000-P', 'BTC-260601-74500-C', 'BTC-260601-74500-P', 'BTC-260601-75000-C', 'BTC-260601-75000-P')
wss://fstream.binance.com/public/stream?streams=btc-260601-73000-c@depth5@500ms/btc-260601-73000-p@depth5@500ms/btc-260601-73500-c@depth5@500ms/btc-260601-73500-p@depth5@500ms/btc-260601-74000-c@depth5@500ms/btc-260601-74000-p@depth5@500ms/btc-260601-74500-c@depth5@500ms/btc-260601-74500-p@depth5@500ms/btc-260601-75000-c@depth5@500ms/btc-260601-75000-p@depth5@500ms


## 4. Start The Background Receiver

This opens one real Binance Options WebSocket connection. The notebook starts without `block_until_ready=True` and then polls connection-level readiness. Some option contracts can still be missing snapshots after the service is ready, so use the latest table and `get_latest(symbol)` to check per-contract availability.

In [4]:
service.start(block_until_ready=False)

deadline = time.time() + 150
while time.time() < deadline and not service.is_ready:
    status = service.status()
    print(
        f"waiting... ready={status.ready} connected={status.connected} "
        f"snapshots={len(service.get_all_latest())}/{len(status.symbols)} "
        f"last_error={status.last_error}",
        end="\r",
    )
    time.sleep(1)

print()
status = service.status()
print("ready:", status.ready)
print("connected:", status.connected)
print("snapshots:", len(service.get_all_latest()), "/", len(status.symbols))
print("last_error:", status.last_error)

waiting... ready=False connected=False snapshots=0/10 last_error=None
ready: True
connected: True
snapshots: 2 / 10
last_error: None


## 5. Convert Latest Snapshots To A Table

Options books may have fewer than 5 levels even when subscribing to `depth5`. `get_latest_level_quote()` returns `None` for a missing or stale snapshot, and returns `None` price/qty fields for a missing bid or ask side.

In [5]:
def quote_side_to_dict(quote, side):
    if quote is None:
        return {"price": None, "qty": None}
    return {
        "price": quote.value(side, "price"),
        "qty": quote.value(side, "qty"),
    }


def snapshot_to_row(snapshot):
    level1 = service.get_latest_level_quote(snapshot.symbol, level=1)
    level2 = service.get_latest_level_quote(snapshot.symbol, level=2)

    bid1_dict = quote_side_to_dict(level1, "bid")
    bid2_dict = quote_side_to_dict(level2, "bid")
    ask1_dict = quote_side_to_dict(level1, "ask")
    ask2_dict = quote_side_to_dict(level2, "ask")

    return {
        "symbol": snapshot.symbol,
        "event_time": pd.to_datetime(snapshot.event_time_ms, unit="ms", utc=True),
        "latency_ms": snapshot.receive_latency_ms,
        "bid1_price": bid1_dict["price"],
        "bid1_qty": bid1_dict["qty"],
        "bid2_price": bid2_dict["price"],
        "bid2_qty": bid2_dict["qty"],
        "ask1_price": ask1_dict["price"],
        "ask1_qty": ask1_dict["qty"],
        "ask2_price": ask2_dict["price"],
        "ask2_qty": ask2_dict["qty"],
        "spread": snapshot.spread,
        "spread_bps": snapshot.spread_bps,
        "bids": len(snapshot.bids),
        "asks": len(snapshot.asks),
        "incomplete": snapshot.depth_incomplete,
        "stale": snapshot.is_stale,
        "sequence_gap": snapshot.sequence_gap,
    }


def latest_depth_frame(service):
    rows = [snapshot_to_row(snapshot) for snapshot in service.get_all_latest().values()]
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).sort_values("symbol").reset_index(drop=True)


latest_df = latest_depth_frame(service)
display(latest_df)

,symbol,event_time,latency_ms,bid1_price,bid1_qty,bid2_price,bid2_qty,ask1_price,ask1_qty,ask2_price,ask2_qty,spread,spread_bps,bids,asks,incomplete,stale,sequence_gap
0,BTC-260601-73000-P,2026-05-31 13:17:00.402000+00:00,45,140.000,0.96,135.000,5.93,145.000,87.68,150.000,0.98,5.000,350.8771929824561403508771930,5,3,True,False,False
1,BTC-260601-73500-P,2026-05-31 13:17:02.478000+00:00,45,230.000,5.90,225.000,6.34,235.000,13.61,240.000,10.95,5.000,215.0537634408602150537634409,5,5,False,False,False
2,BTC-260601-74000-C,2026-05-31 13:17:03.633000+00:00,46,370.000,0.72,365.000,3.85,375.000,1.43,380.000,3.39,5.000,134.2281879194630872483221477,5,5,False,False,False
3,BTC-260601-74000-P,2026-05-31 13:17:03.539000+00:00,45,430.000,1.05,420.000,0.53,435.000,41.18,440.000,6.85,5.000,115.6069364161849710982658960,5,5,False,False,False
4,BTC-260601-74500-C,2026-05-31 13:17:03.452000+00:00,45,155.000,50.18,150.000,8.77,160.000,6.12,165.000,16.15,5.000,317.4603174603174603174603175,5,5,False,False,False
5,BTC-260601-75000-C,2026-05-31 13:17:03.652000+00:00,46,60.000,26.06,55.000,38.71,65.000,21.07,70.000,41.59,5.000,800.00,5,5,False,False,False


## 6. Monitor The Latest Table Briefly

This cell refreshes the display a few times. Re-run it whenever you want to inspect fresh snapshots.

In [6]:
for _ in range(5):
    clear_output(wait=True)
    status = service.status()
    print(
        "running:", status.running,
        "ready:", status.ready,
        "connected:", status.connected,
        "reconnect_attempts:", status.reconnect_attempts,
        "last_error:", status.last_error,
    )
    display(latest_depth_frame(service))
    time.sleep(2)

running: True ready: True connected: True reconnect_attempts: 1 last_error: None


,symbol,event_time,latency_ms,bid1_price,bid1_qty,bid2_price,bid2_qty,ask1_price,ask1_qty,ask2_price,ask2_qty,spread,spread_bps,bids,asks,incomplete,stale,sequence_gap
0,BTC-260601-73000-P,2026-05-31 13:17:25.742000+00:00,45,140.000,4.96,135.000,1.93,145.000,87.68,150.000,0.98,5.000,350.8771929824561403508771930,5,3,True,False,False
1,BTC-260601-73500-C,2026-05-31 13:17:25.126000+00:00,47,680.000,20.00,615.000,3.00,700.000,3.00,705.000,26.60,20.000,289.8550724637681159420289855,5,4,True,False,False
2,BTC-260601-73500-P,2026-05-31 13:17:25.131000+00:00,46,230.000,13.48,225.000,26.29,235.000,13.41,240.000,8.49,5.000,215.0537634408602150537634409,5,5,False,False,False
3,BTC-260601-74000-C,2026-05-31 13:17:25.578000+00:00,46,370.000,0.94,365.000,20.03,375.000,7.79,380.000,3.39,5.000,134.2281879194630872483221477,5,5,False,False,False
4,BTC-260601-74000-P,2026-05-31 13:17:25.636000+00:00,46,430.000,4.05,425.000,1.50,435.000,37.18,440.000,9.45,5.000,115.6069364161849710982658960,5,5,False,False,False
5,BTC-260601-74500-C,2026-05-31 13:17:25.313000+00:00,45,155.000,41.85,150.000,5.22,160.000,10.08,165.000,16.16,5.000,317.4603174603174603174603175,5,5,False,False,False
6,BTC-260601-74500-P,2026-05-31 13:17:24.603000+00:00,45,695.000,1.00,690.000,2.60,800.000,2.60,None,None,105.000,1404.682274247491638795986622,5,1,True,False,False
7,BTC-260601-75000-C,2026-05-31 13:17:25.638000+00:00,46,60.000,25.48,55.000,22.87,65.000,52.95,70.000,9.71,5.000,800.00,5,5,False,False,False
8,BTC-260601-75000-P,2026-05-31 13:17:24.577000+00:00,45,1080.000,2.60,990.000,0.23,1210.000,2.60,1650.000,0.01,130.000,1135.371179039301310043668122,4,2,True,False,False


## 7. Stop The Background Thread

Always stop the service when you are done with the notebook kernel or before re-running the start cell.

In [7]:
if "service" in globals():
    service.stop(timeout=10.0)
    display(service.status())

OptionsDepthServiceStatus(symbols=('BTC-260531-76000-C', 'BTC-260531-76000-P', 'BTC-260531-77000-C', 'BTC-260531-77000-P', 'BTC-260531-78000-C', 'BTC-260531-78000-P'), running=False, ready=False, connected=False, url='wss://fstream.binance.com/public/stream?streams=btc-260531-76000-c@depth5@100ms/btc-260531-76000-p@depth5@100ms/btc-260531-77000-c@depth5@100ms/btc-260531-77000-p@depth5@100ms/btc-260531-78000-c@depth5@100ms/btc-260531-78000-p@depth5@100ms', last_connect_at=datetime.datetime(2026, 5, 31, 7, 14, 57, 448485, tzinfo=datetime.timezone.utc), last_message_at=datetime.datetime(2026, 5, 31, 7, 16, 6, 444917, tzinfo=datetime.timezone.utc), reconnect_attempts=1, last_error=None)